# 🍯 AI Honeypot Security System
**Notebook**: End-to-end pipeline — log ingestion → Gemini AI analysis → threat-intel probes → automated IP blocking.

| Step | Cell |
|------|------|
| 1 | Install dependencies |
| 2 | Configure Gemini |
| 3 | Analyse Cowrie (SSH) logs |
| 4 | Analyse HTTP logs |
| 5 | Threat-intel probes |
| 6 | IP blocking |


In [ ]:
# ── Cell 1 · Install dependencies ───────────────────────────────────────────
!pip install -q --upgrade google-generativeai requests

In [ ]:
# ── Cell 2 · Imports & Gemini configuration ──────────────────────────────────
import os, json, sys
from pprint import pprint

# Add project root to path so local modules resolve correctly
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from ai_engine.gemini_analyzer import (
    configure_gemini,
    analyze_cowrie_chunks,
    analyze_http_chunks,
    aggregate_reports,
)
from threat_intel.probes import run_full_probes, extract_ips_from_file
from threat_intel.geoip  import print_probe_summary
from response_engine.ip_blocker_client import prompt_and_block

# Detect Colab
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("Detected Google Colab.")
except ImportError:
    colab_files = None
    IN_COLAB = False
    print("Local runtime.")

# ─── Configure API key ────────────────────────────────────────────────────────
# Best practice: set GEMINI_API_KEY as an environment variable rather than
# hard-coding it here (or use Colab Secrets).
API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
MODEL   = configure_gemini(API_KEY)

## 🐄 Step 3 — Cowrie (SSH honeypot) log analysis

In [ ]:
# ── Cell 3a · Load cowrie.json ────────────────────────────────────────────────
if IN_COLAB:
    print("Upload your cowrie.json file:")
    uploaded = colab_files.upload()
    cowrie_path = next(iter(uploaded))
else:
    cowrie_path = os.path.join(PROJECT_ROOT, "honeypot", "logs", "cowrie.json")

print("Using:", cowrie_path)

In [ ]:
# ── Cell 3b · Parse & structure cowrie entries ────────────────────────────────
def parse_cowrie_log(path):
    """Parse newline-delimited JSON (or a single JSON array)."""
    entries = []
    with open(path, "r", encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                entries.append(json.loads(line))
            except json.JSONDecodeError:
                try:   # maybe the whole file is one JSON array
                    fh.seek(0)
                    arr = json.load(fh)
                    if isinstance(arr, list):
                        entries.extend(arr)
                    break
                except Exception:
                    pass

    def _extract(o):
        return {
            "timestamp": o.get("timestamp", o.get("time", "")),
            "src_ip":    o.get("src_ip", o.get("peer", {}).get("host", "")),
            "event":     o.get("event",   o.get("message", "")),
            "input":     o.get("input",   o.get("input_data", "")),
            "username":  o.get("username", ""),
            "session":   o.get("session",  ""),
        }

    return [_extract(o) for o in entries]

cowrie_entries = parse_cowrie_log(cowrie_path)
print(f"Parsed {len(cowrie_entries)} cowrie entries.")
for e in cowrie_entries[:3]:
    pprint(e)

In [ ]:
# ── Cell 3c · Run Gemini analysis on cowrie entries ───────────────────────────
COWRIE_CHUNK_SIZE = 60   # tune: 30–100 entries per LLM call

cowrie_reports = analyze_cowrie_chunks(
    cowrie_entries,
    model_name  = MODEL,
    chunk_size  = COWRIE_CHUNK_SIZE,
    output_dir  = os.path.join(PROJECT_ROOT, "reports", "cowrie"),
)

cowrie_agg = aggregate_reports(
    cowrie_reports,
    output_dir = os.path.join(PROJECT_ROOT, "reports"),
    label      = "cowrie",
)

if cowrie_reports:
    print("\n=== First chunk sample ===\n")
    print(cowrie_reports[0][1][:1500])

## 🌐 Step 4 — HTTP log analysis

In [ ]:
# ── Cell 4a · Load HTTP log ───────────────────────────────────────────────────
if IN_COLAB:
    print("Upload your HTTP log file (JSON-lines or Apache/Nginx combined format):")
    uploaded = colab_files.upload()
    http_log_path = next(iter(uploaded))
else:
    http_log_path = os.path.join(PROJECT_ROOT, "honeypot", "logs", "http.log")

print("Using:", http_log_path)

In [ ]:
# ── Cell 4b · Parse & enrich HTTP entries ─────────────────────────────────────
import re, urllib.parse

COMBINED_LOG_RE = re.compile(
    r'(?P<ip>\S+)\s+(?P<ident>\S+)\s+(?P<authuser>\S+)\s+'
    r'\[(?P<time>[^\]]+)\]\s+"(?P<request>[^"]+)"\s+'
    r'(?P<status>\d{3})\s+(?P<size>\S+)'
    r'(?:\s+"(?P<referrer>[^"]*)"\s+"(?P<agent>[^"]*)")?'
)

def _parse_apache_time(t):
    from datetime import datetime
    try:
        return datetime.strptime(t, "%d/%b/%Y:%H:%M:%S %z").isoformat()
    except Exception:
        return t

def _parse_http_line(line):
    line = line.strip()
    if not line:
        return None
    # 1) JSON
    try:
        obj = json.loads(line)
        get = lambda *ks: next((obj[k] for k in ks if k in obj and obj[k] is not None), "")
        return {
            "timestamp": get("timestamp","time","ts","@timestamp"),
            "src_ip":    get("src_ip","remote_addr","client_ip","ip"),
            "method":    get("method","http_method"),
            "path":      get("path","request","url","request_uri"),
            "status":    str(get("status","status_code","response")),
            "size":      str(get("bytes","size","body_bytes_sent")),
            "referrer":  get("referrer","referer"),
            "user_agent":get("user_agent","agent","http_user_agent"),
            "protocol":  get("protocol","http_version"),
        }
    except json.JSONDecodeError:
        pass
    # 2) Combined log
    m = COMBINED_LOG_RE.match(line)
    if m:
        req = m.group("request") or ""
        parts = req.split()
        return {
            "timestamp": _parse_apache_time(m.group("time") or ""),
            "src_ip":    m.group("ip"),
            "method":    parts[0] if parts else "",
            "path":      parts[1] if len(parts) > 1 else "",
            "protocol":  parts[2] if len(parts) > 2 else "",
            "status":    m.group("status"),
            "size":      m.group("size"),
            "referrer":  m.group("referrer") or "",
            "user_agent":m.group("agent") or "",
        }
    return None

def _enrich(e, line_no):
    p  = (e.get("path") or "").lower()
    ua = (e.get("user_agent") or "").lower()
    flags = []
    if any(x in p for x in [".php",".env","wp-login","shell","cmd=","eval(","base64"]):
        flags.append("suspicious_path")
    if "union select" in p or ("select " in p and "from " in p):
        flags.append("sql_injection")
    if any(t in ua for t in ["wget","curl","sqlmap","nikto","masscan"]):
        flags.append("scanner_ua")
    e["path_normalized"]  = urllib.parse.unquote(e.get("path",""))
    e["suspicion_flags"]  = flags
    e["_line_no"]         = line_no
    return e

http_entries = []
with open(http_log_path, "r", encoding="utf-8", errors="ignore") as fh:
    for ln, line in enumerate(fh, 1):
        parsed = _parse_http_line(line)
        if parsed:
            http_entries.append(_enrich(parsed, ln))

print(f"Parsed {len(http_entries)} HTTP entries.")
for e in http_entries[:3]:
    pprint(e)

In [ ]:
# ── Cell 4c · Run Gemini analysis on HTTP entries ─────────────────────────────
HTTP_CHUNK_SIZE = 50

http_reports = analyze_http_chunks(
    http_entries,
    model_name = MODEL,
    chunk_size = HTTP_CHUNK_SIZE,
    output_dir = os.path.join(PROJECT_ROOT, "reports", "http"),
)

http_agg = aggregate_reports(
    http_reports,
    output_dir = os.path.join(PROJECT_ROOT, "reports"),
    label      = "http",
)

if http_reports:
    print("\n=== First chunk sample ===\n")
    print(http_reports[0][1][:1500])

## 🔍 Step 5 — Threat-intel probes

In [ ]:
# ── Cell 5 · Extract IPs from the aggregated AI report & probe them ──────────
import os

REPORT_TO_PROBE = cowrie_agg   # or http_agg, or any other report path

ips = extract_ips_from_file(REPORT_TO_PROBE)
print("IPs found:", ips)

probe_results = {}
for ip in ips:
    summary = run_full_probes(ip)
    out_json = os.path.join(PROJECT_ROOT, "reports", f"probe_{ip}.json")
    print_probe_summary(summary, outpath=out_json)
    probe_results[ip] = summary

## 🚫 Step 6 — Automated IP blocking

In [ ]:
# ── Cell 6 · Send block requests to the response_engine socket server ─────────
# Make sure socket_server.py is running on the target host before executing.
RECEIVER_HOST = "192.168.46.19"   # ← change to your honeypot host
RECEIVER_PORT = 5002

for ip in ips:
    prompt_and_block(ip, receiver_host=RECEIVER_HOST, receiver_port=RECEIVER_PORT)

In [ ]:
# ── Cell 7 · Download aggregated reports (Colab only) ────────────────────────
if IN_COLAB:
    for path in [cowrie_agg, http_agg]:
        if os.path.exists(path):
            colab_files.download(path)
            print("Downloading:", path)
else:
    print("Reports saved locally:")
    for path in [cowrie_agg, http_agg]:
        print(" ", os.path.abspath(path))